# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access metadata properties
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Published: {md.datePublished}\nLicense: {md.license}")
print(f"Keywords: {md.keywords}\nIdentifier: {md.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

> In Croissant, record sets represent tables or logical groupings of records. Every entity is referenced by its `@id`.

Let's enumerate available record sets and their associated fields.

In [ ]:
# Fetch record set metadata by @id
record_sets = dataset.metadata.recordSets  # list of mlcroissant.RecordSet
if hasattr(record_sets, '__iter__'):
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"Record Set: {rs.name} | @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'dataType', 'unknown')})")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract all available record sets into pandas DataFrames, referencing each by their `@id`.

Choose the appropriate record set `@id` for further processing (for example, demographic survey responses).

In [ ]:
# Identify all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print columns for the first record set
first_rs = record_set_ids[0] if record_set_ids else None
if first_rs:
    print(f"Columns for {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All field references below use the `@id` as a column name (as per Croissant conventions).

In [ ]:
# Choose a record set (use the first one for this example)
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# List numeric fields from metadata
numeric_fields = []
rs_metadata = None
for rs in dataset.metadata.recordSets:
    if rs.id == record_set_id:
        rs_metadata = rs
        break
if rs_metadata:
    numeric_fields = [f.id for f in rs_metadata.fields if getattr(f, 'dataType', '').lower() in ['integer', 'float', 'number']]

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Selected numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if not df.empty else 0
    # Filtering: records with value above mean
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field
    group_field = None
    for f in rs_metadata.fields:
        if getattr(f, 'dataType', 'unknown').lower() in ['text', 'string', 'boolean', 'date']:
            group_field = f.id
            break
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the chosen numeric field and a grouped summary by a categorical field. All references use the `@id`.

In [ ]:
import matplotlib.pyplot as plt

if not df.empty and numeric_fields:
    numeric_field = numeric_fields[0]
    plt.figure(figsize=(7, 4))
    df[numeric_field].dropna().hist(bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped_df exists
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field], grouped_df[numeric_field], color='orange')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the Croissant specification and explored its structure with `mlcroissant`. By referencing entities via their `@id`, we performed filtering, normalization, grouping, and visualization of key indicators. The dataset enables further research and analysis of mental health patterns based on standardized survey responses.

For deeper insights, continue investigating additional record sets and fields referenced by their `@id`.